In [10]:
import os
import shutil
import kagglehub
from ultralytics import YOLO
from tqdm import tqdm
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
path = kagglehub.dataset_download('adilshamim8/people-detection')

YOLO_EVAL_DATA = './data'
os.makedirs(YOLO_EVAL_DATA, exist_ok=True)

for name in os.listdir(path):
    s = os.path.join(path, name)
    d = os.path.join(YOLO_EVAL_DATA, name)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

In [4]:
def voc_to_yolo_xywh_norm(xmin, ymin, xmax, ymax, img_w, img_h):
    # centers
    x_c = (xmin + xmax) / 2.0
    y_c = (ymin + ymax) / 2.0
    # width/height
    w = xmax - xmin
    h = ymax - ymin
    # normalize
    return x_c / img_w, y_c / img_h, w / img_w, h / img_h

In [5]:
CSV_PATH = ''
for root, _, files in os.walk('./data/train/train'):
  for file in files:
      if file.endswith(".csv"):
          CSV_PATH = os.path.join(root, file)
          
IMAGES_SRC_DIR = "./data/train/train"  # where the images actually live

df = pd.read_csv(CSV_PATH)

In [6]:
# Deterministic split
rng = np.random.default_rng(42)
unique_imgs = df['filename'].unique()
rng.shuffle(unique_imgs)

val_ratio = 0.2
val_count = int(len(unique_imgs) * val_ratio)
val_set = set(unique_imgs[:val_count])
train_set = set(unique_imgs[val_count:])

len(unique_imgs), len(train_set), len(val_set)

C:\Users\abdel\AppData\Local\Temp\ipykernel_1376\3271254232.py:4: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(unique_imgs)


(13278, 10623, 2655)

In [ ]:
IMAGES_SRC_DIR = "./data/train/train"
missing_images = []
label_counts = {"train": 0, "val": 0}

def resolve_image_path(images_dir, name):
    p = Path(images_dir) / name
    if p.exists():
        return p
    candidates = [
        p.with_suffix(".jpg"),
        p.with_suffix(".jpeg"),
        p.with_suffix(".png"),
        p.with_suffix(".JPG"),
        p.with_suffix(".PNG"),
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

for filename, group in tqdm(df.groupby("filename"), total=df['filename'].nunique()):
    subset = "val" if filename in val_set else "train"

    # Resolve input image
    src_img = resolve_image_path(IMAGES_SRC_DIR, filename)
    if src_img is None:
        missing_images.append(filename)
        continue

    # Copy image to working set
    dst_img = Path('data') / 'images' / subset / src_img.name
    dst_img.parent.mkdir(parents=True, exist_ok=True)
    if not dst_img.exists():
        shutil.copy2(src_img, dst_img)

    # Build label path (same stem, .txt)
    dst_lbl = Path('data') / 'labels' / subset / (dst_img.stem + ".txt")
    dst_lbl.parent.mkdir(parents=True, exist_ok=True)

    # Use first record to get image width/height (they should be identical per image)
    img_w = int(group.iloc[0]["width"])
    img_h = int(group.iloc[0]["height"])

    # Write all boxes for this image
    lines = []
    for _, r in group.iterrows():
        xmin, ymin, xmax, ymax = float(r["xmin"]), float(r["ymin"]), float(r["xmax"]), float(r["ymax"])
        x_c, y_c, w, h = voc_to_yolo_xywh_norm(xmin, ymin, xmax, ymax, img_w, img_h)
        # class_id = 0 (only person)
        lines.append(f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

    with open(dst_lbl, "w") as f:
        f.write("\n".join(lines))

    label_counts[subset] += len(lines)

print("Train labels:", label_counts["train"], "| Val labels:", label_counts["val"])

  0%|          | 0/13278 [00:00<?, ?it/s]

100%|██████████| 13278/13278 [00:48<00:00, 273.23it/s]

Train labels: 79098 | Val labels: 20984


In [15]:
model = YOLO('yolo11n.pt')

Evaluation graphs (precision/recall/loss, etc.) are saved to the runs/detect/valx directory — open that folder to view the generated plots.

In [17]:
metrics = model.val(data="./data.yaml", plots=True)
metrics.results_dict

Ultralytics 8.4.7  Python-3.13.7 torch-2.9.1+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
val: Fast image access  (ping: 0.20.0 ms, read: 9.48.5 MB/s, size: 137.8 KB)
val: Scanning D:\Studies\AI Models_Applications\FINAL\Person_Detection_and_Face_Recongnition\eval\yolo_eval_Data\data\labels\val... 2655 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2655/2655 165.8it/s 16.0s<0.0s
val: D:\Studies\AI Models_Applications\FINAL\Person_Detection_and_Face_Recongnition\eval\yolo_eval_Data\data\images\val\2007_000187_jpg.rf.6072bb3923494a7afb8648ba483324fd.jpg: 23 duplicate labels removed
val: New cache created: D:\Studies\AI Models_Applications\FINAL\Person_Detection_and_Face_Recongnition\eval\yolo_eval_Data\data\labels\val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 166/166 1.8s/it 5:051.8ss
                   all       2655      20961      0.713        0.5      0.544      0.293
                person       2655  

{'metrics/precision(B)': 0.7132130959742258,
 'metrics/recall(B)': 0.4997310774283084,
 'metrics/mAP50(B)': 0.5441442471992685,
 'metrics/mAP50-95(B)': 0.29348096803468143,
 'fitness': 0.29348096803468143}